In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
import scienceplots
import scipy
from dotenv import load_dotenv
import os
import pyvista as pv

import tensorstore as ts

load_dotenv()
PATH = os.getenv("ROOT_PATH")

plt.style.use(['science', 'no-latex'])

def format_ax(ax):
  for spine in ax.spines.values():
    spine.set_linewidth(1.2)
  ax.spines['top'].set_visible(False)
  ax.spines['right'].set_visible(False)
  ax.spines['bottom'].set_visible(False)
  ax.tick_params(which='minor', length=0)
  ax.tick_params(axis='both', labelsize=12)
  for spine in ax.spines.values():
    spine.set_visible(False)
  ax.tick_params(axis='both', which='both', bottom=False, top=False, left=False, right=False, labelbottom=False, labelleft=False)
  ax.tick_params(axis='y', which='both', left=False, right=False, direction="out", width=1.2)

In [ ]:
subject_id = "06"
traces = ts.open({
    'open': True,
    'driver': 'zarr3',
    'kvstore': f'file://{PATH}/ts_files/subject_{subject_id}_traces.zarr'
    # 'kvstore': 'gs://zapbench-release/volumes/20240930/traces/'
}).result()

s = ts.open({
    'open': True,
    'driver': 'zarr3',
    'kvstore': f'file://{PATH}/ts_files/subject_{subject_id}_stimuli.zarr'
    # 'kvstore': 'gs://zapbench-release/volumes/20240930/traces/'
}).result()

coordinates = ts.open({
    'open': True,
    'driver': 'zarr3',
    'kvstore': f'file://{PATH}/ts_files/subject_{subject_id}_coordinates.zarr'
    # 'kvstore': 'gs://zapbench-release/volumes/20240930/traces/'
}).result()

behavior = ts.open({
    'open': True,
    'driver': 'zarr3',
    'kvstore': f'file://{PATH}/ts_files/subject_{subject_id}_behavioral_covariates.zarr'
    # 'kvstore': 'gs://zapbench-release/volumes/20240930/traces/'
}).result()

traces = traces.read().result()
s = s.read().result()
coordinates = coordinates.read().result()
behavior = behavior.read().result()

In [ ]:
traces.shape, s.shape, coordinates.shape, behavior.shape

In [ ]:
import h5py
import numpy as np
f = h5py.File(f'{PATH}/Additional_mat_files/MaskDatabase.mat', 'r')
names = f['MaskDatabaseNames']
names = [(i, "".join([chr(c[0]) for c in f[name[0]]])) for i, name in enumerate(names)]
names

In [ ]:
def linear_to_3d_matlab(linear_idx, width, height):
  idx = linear_idx - 1
  i = idx % width
  j = (idx // width) % height
  k = idx // (width * height)
  return i, j, k

def matlab_to_linear(i, j, k, width, height):
  linear_idx = i + j * width + k * width * height + 1
  return linear_idx

Find neurons that correlate with behavior

In [ ]:
fig, axs = plt.subplots(5, figsize=(10, 3), dpi=200)
for i in range(0, 5):
  ax = axs[i]
  ax.plot(behavior[..., i],'k',linewidth=2)
  format_ax(ax)
plt.tight_layout()
plt.show()
fig, axs = plt.subplots(5, figsize=(10, 3), dpi=200)
for i in range(0, 5):
  ax = axs[i]
  ax.plot(np.where(s==1)[-1],'k',linewidth=2)
  format_ax(ax)
plt.tight_layout()
plt.show()

In [ ]:
from tqdm import tqdm
trace_i = np.where(s==1)[-1]
cfs = []
for j in tqdm(range(traces.shape[1])):
  cfs.append(np.corrcoef(traces[..., j], trace_i)[0, 1])

In [ ]:
traces_c = traces[..., np.argsort(np.abs(cfs))[::-1][:20]]

In [ ]:
fig, axs = plt.subplots(20, figsize=(10, 10), dpi=200)
for i in range(0, 20):
  ax = axs[i]
  ax.plot(traces_c[..., i],'k',linewidth=2)
  ax.plot(behavior[..., 0],'r',alpha=0.3,linewidth=1)
  format_ax(ax)
plt.tight_layout()
plt.show()

Check where they are

In [ ]:
reference_anat = scipy.io.loadmat(f'{PATH}/Additional_mat_files/ReferenceBrain.mat')['anat_stack_norm']

ir = f['MaskDatabase']['ir'][:]  # Row indices (linear voxel indices)
jc = f['MaskDatabase']['jc'][:]  # Column pointers
data = f['MaskDatabase']['data'][:]  # Should be all ones

mask = np.zeros_like(reference_anat)
mask_shape = mask.shape

grid = pv.wrap(reference_anat)
grid.spacing = [1.0, 1.0, 2.0]

valid_coordinates_correlation = coordinates[np.argsort(np.abs(cfs))[::-1][:30]]
# points = valid_coordinates_correlation * np.array(grid.spacing)
# point_cloud = pv.PolyData(points)
# colors = np.abs(cfs)[np.argsort(np.abs(cfs))[::-1][:100]]

# plotter = pv.Plotter(notebook=True)
# plotter.add_volume(
#     grid,
#     cmap="gray",
#     opacity=[0.0,0.1,0.3,0.5],
#     shade=True,
#     show_scalar_bar=False,
# )

# plotter.add_mesh(
#     point_cloud,
#     scalars=colors,
#     cmap='viridis',
#     point_size=5,
#     render_points_as_spheres=True,
#     show_scalar_bar=False
# )

# plotter.camera_position = 'xy'
# plotter.camera.azimuth = 0
# plotter.camera.elevation = 0
# plotter.camera.zoom(1.5)
# plotter.show()

In [ ]:
mask_idx_list = []
for i_x, i_y, i_z in valid_coordinates_correlation.astype(int):
  i_lin = matlab_to_linear(i_x, i_y, i_z, mask_shape[0], mask_shape[1])
  for ir_idx in np.where(ir == i_lin)[0]:
    mask_idx_list.append(np.where(jc < ir_idx)[0][-1])
mask_idx_list = np.unique(mask_idx_list)
mask_idx_list

In [ ]:
chosen_region_indices = [1, 3, 4, 5, 6, 7]

In [ ]:
np.array(names)[mask_idx_list]

Now locate the regions where the correlated neurons are and check in the atlas if they are anatomically connected

In [ ]:
reference_anat = scipy.io.loadmat(f'{PATH}/Additional_mat_files/ReferenceBrain.mat')['anat_stack_norm']

ir = f['MaskDatabase']['ir'][:]  # Row indices (linear voxel indices)
jc = f['MaskDatabase']['jc'][:]  # Column pointers
data = f['MaskDatabase']['data'][:]  # Should be all ones

mask = np.zeros_like(reference_anat)
mask_shape = mask.shape

In [ ]:
for mask_idx in mask_idx_list:
  start_idx = jc[mask_idx]
  end_idx = jc[mask_idx + 1]
  for i in ir[start_idx:end_idx]:
    i_x, i_y, i_z = linear_to_3d_matlab(i, mask_shape[0], mask_shape[1])
    mask[i_x, i_y, i_z] = 1

In [ ]:
reference_anat[mask==0] = 0
grid = pv.wrap(reference_anat)
grid.spacing = [1.0, 1.0, 2.0]

valid_coordinates = coordinates[np.argsort(np.abs(cfs))[::-1][:30]]
points = valid_coordinates * np.array(grid.spacing)
point_cloud = pv.PolyData(points)
colors = np.abs(cfs)[np.argsort(np.abs(cfs))[::-1][:30]]

plotter = pv.Plotter(notebook=True)
plotter.add_volume(
    grid,
    cmap="gray",
    opacity=[0.0,0.1,0.3,0.5],
    shade=True,
    show_scalar_bar=False,
)

plotter.add_mesh(
    point_cloud,
    scalars=colors,
    cmap='viridis',
    point_size=20,
    render_points_as_spheres=True,
    show_scalar_bar=False
)

plotter.camera_position = 'xy'
plotter.camera.azimuth = 0
plotter.camera.elevation = 0
plotter.camera.zoom(1.5)
plotter.show()

In [ ]:
reference_anat[mask==0] = 0
grid = pv.wrap(reference_anat)
grid.spacing = [1.0, 1.0, 2.0]

valid_coordinates = coordinates[np.argsort(np.abs(cfs))[::-1][:30]]
points = valid_coordinates * np.array(grid.spacing)
point_cloud = pv.PolyData(points)
colors = np.abs(cfs)[np.argsort(np.abs(cfs))[::-1][:30]]

plotter = pv.Plotter(notebook=True)
plotter.add_volume(
    grid,
    cmap="gray",
    opacity=[0.0,0.1,0.3,0.5],
    shade=True,
    show_scalar_bar=False,
)

plotter.add_mesh(
    point_cloud,
    scalars=colors,
    cmap='viridis',
    point_size=20,
    render_points_as_spheres=True,
    show_scalar_bar=False
)

plotter.camera_position = 'xy'
plotter.camera.azimuth = 0
plotter.camera.elevation = 0
plotter.camera.zoom(1.5)
plotter.show()